# SSTZIP-GNN Training Notebook (Method2)

Train SSTZIP-GNN only for Method2 in an isolated session.

**Works on:**
- Local machine (CPU/GPU)
- Google Colab (GPU T4/A100)

**Steps:**
1. Setup environment & check GPU
2. Load configuration (force Method2)
3. Import project modules
4. Run smoke check + training for Method2
5. Display and sync Method2 results

## 1. Environment Setup

In [ ]:
# Check if running on Colab
import sys
import os
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
    print("[INFO] Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("[INFO] Running locally")

# Setup environment
if IN_COLAB:
    # Mount Google Drive
    drive.mount('/content/drive', force_remount=False)
    print("[OK] Google Drive mounted")
    
    # Clone repository
    print("[Clone] Cloning repository...")
    !git clone -b develop https://github.com/senkochi/taxi-demand-prediction.git /content/taxi-demand-prediction 2>/dev/null || echo "Repository already cloned"
    
    os.chdir('/content/taxi-demand-prediction')
    print(f"[OK] Working directory: {os.getcwd()}")
    
    # Link data from Google Drive
    print("\n[Link] Linking data from Google Drive...")
    drive_data = Path('/content/drive/MyDrive/data')
    local_data = Path('data')
    
    if not local_data.exists():
        try:
            os.symlink(drive_data, 'data')
            print(f"  [OK] Symlink: ./data → {drive_data}")
        except (OSError, NotImplementedError):
            import shutil
            shutil.copytree(drive_data, 'data')
            print(f"  [OK] Data copied from Google Drive")
    else:
        print(f"  [OK] ./data already exists")
else:
    # Local setup - navigate to project root
    notebook_dir = Path.cwd()
    
    # If in notebooks folder, go up to project root
    if notebook_dir.name == 'notebooks':
        project_root = notebook_dir.parent
    elif (notebook_dir / 'notebooks').exists():
        project_root = notebook_dir
    else:
        project_root = notebook_dir.parent
    
    os.chdir(project_root)
    print(f"[OK] Working directory: {os.getcwd()}")

# Verify structure
print("\n[Check] Project structure:")
for item in ['src', 'config', 'scripts', 'data']:
    exists = Path(item).exists()
    status = "✓" if exists else "✗"
    print(f"  {status} {item}/")

[INFO] Running on Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] Google Drive mounted
[Clone] Cloning repository...
Repository already cloned
[OK] Working directory: /content/taxi-demand-prediction

[Link] Linking data from Google Drive...
  [OK] ./data already exists

[Check] Project structure:
  ✓ src/
  ✓ config/
  ✓ scripts/
  ✓ data/


In [ ]:
# Install dependencies on Colab
if IN_COLAB:
    print("[Install] Installing PyTorch Lightning...")
    !pip install pytorch-lightning -q
    !pip install duckdb -q
    !pip install pyyaml -q
    print("[OK] Dependencies installed")

[Install] Installing PyTorch Lightning...
[OK] Dependencies installed


In [ ]:
# Check GPU availability
import torch

print("[System] PyTorch environment:")
print(f"  - Version: {torch.__version__}")
print(f"  - CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  - GPU: {torch.cuda.get_device_name(0)}")
    print(f"  - VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    print(f"  - Using CPU")
    device = 'cpu'

[System] PyTorch environment:
  - Version: 2.11.0+cpu
  - CUDA available: False
  - Using CPU


## 2. Load Configuration

In [ ]:
import yaml
from pathlib import Path
import os

TARGET_METHOD = 'method2'

cwd = os.getcwd()
print(f"[Debug] Current working directory: {cwd}\n")

config_paths = [
    Path('config/config.yaml'),
    Path.cwd() / 'config' / 'config.yaml',
]

if Path.cwd().name == 'notebooks' or 'notebooks' in str(Path.cwd()):
    project_root = Path.cwd().parent
    config_paths.insert(0, project_root / 'config' / 'config.yaml')
    print(f"[Debug] Detected notebooks folder, also checking: {config_paths[0]}\n")

config_path = None
for path in config_paths:
    if path.exists():
        config_path = path
        print(f"[OK] Found config at: {path}")
        break

if not config_path:
    print(f"[ERROR] Config not found in any of these locations:")
    for path in config_paths:
        print(f"  - {path}")
    raise FileNotFoundError("Config file not found")

with open(config_path) as f:
    config = yaml.safe_load(f)

config.setdefault('clustering', {})['methods'] = [TARGET_METHOD]

print("\n[Config] Loaded configuration:")
print(f"  - Target method: {TARGET_METHOD}")
print(f"  - Architecture: {config.get('model', {}).get('architecture', 'SSTZIP-GNN')}")
print(f"  - Methods: {config.get('clustering', {}).get('methods', [])}")
print(f"  - Time buckets: {config.get('temporal_aggregation', {}).get('buckets', [])}")
print(f"  - Batch size: {config.get('model', {}).get('training', {}).get('batch_size', 64)}")
print(f"  - Epochs: {config.get('model', {}).get('training', {}).get('epochs', 100)}")
print(f"  - Learning rate: {config.get('model', {}).get('training', {}).get('learning_rate', 0.001)}")

[Debug] Current working directory: /content/taxi-demand-prediction

[OK] Found config at: config/config.yaml

[Config] Loaded configuration:
  - Architecture: SSTZIP-GNN
  - Methods: ['method1', 'method2', 'method3']
  - Time buckets: [15, 30, 60]
  - Batch size: 64
  - Epochs: 100
  - Learning rate: 0.001


## 3. Import Project Modules

In [ ]:
import sys
from pathlib import Path

# Add project to path
sys.path.insert(0, str(Path.cwd()))

# Import project modules
try:
    from src.data.data_loader import TaxiDemandDataModule
    from src.models.sstzip_gnn import SSTZIPGNNModel
    from src.training.trainer import SSTZIPGNNLightning
    from src.evaluation.metrics import Metrics
    print("[OK] All project modules imported successfully!")
except ImportError as e:
    print(f"[ERROR] Failed to import project modules: {e}")
    print("\nAvailable modules in src/:")
    import os
    for item in os.listdir('src'):
        print(f"  - {item}")
    raise

[OK] All project modules imported successfully!


## 4. Verify Required Data Files

In [ ]:
from pathlib import Path

print("[Check] Verifying required data files...\n")

# Check DuckDB features file
duckdb_path = Path('data/processed/taxi_features.duckdb')
if duckdb_path.exists():
    size_mb = duckdb_path.stat().st_size / 1e6
    print(f"  [OK] DuckDB: {duckdb_path} ({size_mb:.1f} MB)")
else:
    print(f"  [MISSING] DuckDB: {duckdb_path}")

# Check cluster assignment files
print("\n[Check] Cluster assignment files:")
cluster_files = {
    'Baseline': Path('data/models/baseline_clusters.pkl'),
    'Method1': Path('data/models/method1_clusters.pkl'),
    'Method2': Path('data/models/method2_clusters.pkl'),
    'Method3': Path('data/models/method3_clusters.pkl'),
}

missing_clusters = []
for name, path in cluster_files.items():
    if path.exists():
        print(f"  [OK] {name}: {path}")
    else:
        print(f"  [MISSING] {name}: {path}")
        missing_clusters.append(name)

if missing_clusters:
    print(f"\n[WARN] Missing cluster files for: {', '.join(missing_clusters)}")
    print("  These should be generated by Phase 3 (clustering scripts)")
else:
    print("\n[OK] All cluster files present!")

print("\n[Note] Data structure expected:")
print("  My Drive/data/")
print("    ├── processed/")
print("    │   ├── taxi_features.duckdb")
print("    │   ├── taxi_features_*.parquet")
print("    │   └── *_report.json")
print("    └── models/")
print("        └── sstzip_gnn/")
print("            ├── baseline_clusters.pkl")
print("            ├── method1_clusters.pkl")
print("            ├── method2_clusters.pkl")
print("            └── method3_clusters.pkl")

[Check] Verifying required data files...

  [OK] DuckDB: data/processed/taxi_features.duckdb (489.7 MB)

[Check] Cluster assignment files:
  [MISSING] Baseline: data/models/baseline_clusters.pkl
  [OK] Method1: data/models/method1_clusters.pkl
  [OK] Method2: data/models/method2_clusters.pkl
  [OK] Method3: data/models/method3_clusters.pkl

[WARN] Missing cluster files for: Baseline
  These should be generated by Phase 3 (clustering scripts)

[Note] Data structure expected:
  My Drive/data/
    ├── processed/
    │   ├── taxi_features.duckdb
    │   ├── taxi_features_*.parquet
    │   └── *_report.json
    └── models/
        └── sstzip_gnn/
            ├── baseline_clusters.pkl
            ├── method1_clusters.pkl
            ├── method2_clusters.pkl
            └── method3_clusters.pkl


In [ ]:
# One-batch smoke check (no full training)
import importlib
import torch

print("[Smoke] Running one-batch pipeline check...")

importlib.invalidate_caches()
import src.data.data_loader as data_loader_module
import src.training.trainer as trainer_module_lib
import src.models.sstzip_gnn as model_module_lib

importlib.reload(data_loader_module)
importlib.reload(trainer_module_lib)
importlib.reload(model_module_lib)

TaxiDemandDataModule = data_loader_module.TaxiDemandDataModule
SSTZIPGNNLightning = trainer_module_lib.SSTZIPGNNLightning
SSTZIPGNNModel = model_module_lib.SSTZIPGNNModel

smoke_method = TARGET_METHOD
smoke_batch_size = min(config.get('model', {}).get('training', {}).get('batch_size', 128), 8)
smoke_num_workers = 0

print(f"[Smoke] Method: {smoke_method}")
print(f"[Smoke] Batch size: {smoke_batch_size}")
print(f"[Smoke] Num workers: {smoke_num_workers}")
print(f"[Smoke] Device: {device}")

smoke_data_module = TaxiDemandDataModule(
    duckdb_path='data/processed/taxi_features.duckdb',
    clustering_method=smoke_method,
    sequence_length=config.get('model', {}).get('seq_length', 96),
    forecast_horizon=1,
    batch_size=smoke_batch_size,
    num_workers=smoke_num_workers,
    train_ratio=config.get('train_val_test', {}).get('train_ratio', 0.85),
    val_ratio=config.get('train_val_test', {}).get('val_ratio', 0.08),
    test_ratio=config.get('train_val_test', {}).get('test_ratio', 0.07)
)
smoke_data_module.setup()

x_smoke, y_smoke = next(iter(smoke_data_module.train_dataloader()))
print(f"[Smoke] x shape: {tuple(x_smoke.shape)}")
print(f"[Smoke] x finite: {torch.isfinite(x_smoke).all().item()}")
print(f"[Smoke] y shape: {tuple(y_smoke.shape)}")
print(f"[Smoke] y finite: {torch.isfinite(y_smoke).all().item()}")

if not torch.isfinite(x_smoke).all():
    raise ValueError("[Smoke] Non-finite values detected in the input batch.")

smoke_model = SSTZIPGNNModel(
    num_zones=smoke_data_module.adjacency_matrix.shape[0],
    feature_dim=x_smoke.shape[-1],
    spatial_dim=config.get('model', {}).get('spatial_dim', 32),
    temporal_dim=config.get('model', {}).get('temporal_dim', 32),
    num_spatial_layers=config.get('model', {}).get('num_spatial_layers', 1),
    num_spatial_hops=config.get('model', {}).get('num_spatial_hops', 2),
    num_temporal_layers=config.get('model', {}).get('num_temporal_layers', 2),
    hidden_dim_zip=config.get('model', {}).get('hidden_dim_zip', 64),
    dropout=config.get('model', {}).get('dropout', 0.1)
).to(device)

smoke_trainer_module = SSTZIPGNNLightning(
    model=smoke_model,
    learning_rate=config.get('model', {}).get('training', {}).get('learning_rate', 0.0003),
    weight_decay=1e-5,
    patience=config.get('model', {}).get('training', {}).get('early_stopping_patience', 5),
    accumulation_steps=1
).to(device)

with torch.no_grad():
    smoke_loss = smoke_trainer_module.training_step((x_smoke.to(device), y_smoke.to(device)), 0)

print(f"[Smoke] training_step loss finite: {torch.isfinite(smoke_loss).item()}")
print(f"[Smoke] loss: {float(smoke_loss.item()):.6f}")
print("[Smoke] One-batch check passed")

## 5. Run Stable Training Script

In [ ]:
# Execute the stable training script for one method
print("="*80)
print(f"EXECUTING: scripts/04_train_model_stable.py --method {TARGET_METHOD}")
print("="*80)
print()

import os
import sys

script_path = os.path.join(os.getcwd(), 'scripts', '04_train_model_stable.py')
with open(script_path, encoding='utf-8') as f:
    training_script = f.read()

original_argv = sys.argv[:]
sys.argv = [script_path, '--method', TARGET_METHOD]
try:
    exec_globals = {'__file__': script_path, '__name__': '__main__'}
    exec(training_script, exec_globals)
finally:
    sys.argv = original_argv

EXECUTING: scripts/04_train_model.py


PHASE 4: DEEP LEARNING MODELING - SSTZIP-GNN TRAINING

[Setup] Loading configuration...

[Data] Initializing data module and loading adjacency matrices...

--------------------------------------------------------------------------------
METHOD: BASELINE
--------------------------------------------------------------------------------
Setting up data module for baseline...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 3513833 feature records from 261 zones

[SKIP] Skipping baseline: Clustering file not found: data/models/baseline_clusters.pkl
   Make sure clustering results exist in data/models/

--------------------------------------------------------------------------------
METHOD: METHOD1
--------------------------------------------------------------------------------
Setting up data module for method1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 3513833 feature records from 261 zones
Adjacency matrix created: torch.Size([264, 264])
Train: 2369537 sequences
Val: 251674 sequences
Test: 868533 sequences
Creating adjacency matrix from method1 clustering...
[OK] Adjacency matrix: torch.Size([264, 264])

TRAINING SSTZIP-GNN: METHOD1

[Config] Sequence length: 96
[Config] Batch size: 64
[Config] Learning rate: 0.001
[Config] Epochs: 100
[1/7] Getting data loaders for method1...
[OK] DataLoaders ready
  - Train batches: 37025
  - Val batches: 3933
  - Test batches: 13571
  - Adjacency matrix: torch.Size([264, 264])

[2/7] Initializing SSTZIP-GNN model...


...

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is 
set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is 
set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[OK] Training complete in 9551.7 seconds

[5/7] Evaluating on test set...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 6. Display and Save Results

In [ ]:
import json
from pathlib import Path

import pandas as pd

print(f"[Results] Loading training summary for {TARGET_METHOD}...\n")

method_metrics_path = Path(f'checkpoints/{TARGET_METHOD}/metrics.json')
summary_path = Path('reports/training_summary.csv')

results_df = None
results = None

if method_metrics_path.exists():
    with open(method_metrics_path, 'r', encoding='utf-8') as f:
        payload = json.load(f)
    results = payload.get('metrics', payload)
    results_df = pd.DataFrame([results], index=[TARGET_METHOD])
    print(f"[OK] Found method metrics at: {method_metrics_path}\n")
elif summary_path.exists():
    results_df = pd.read_csv(summary_path, index_col=0)
    if TARGET_METHOD in results_df.index:
        results_df = results_df.loc[[TARGET_METHOD]]
    results = results_df
    print(f"[OK] Found summary at: {summary_path}\n")
else:
    print("[INFO] No summary file found yet.")

if results_df is not None and not results_df.empty:
    print("="*80)
    print(f"TRAINING RESULTS ({TARGET_METHOD})")
    print("="*80)
    print(results_df.to_string())
elif results:
    print("="*80)
    print(f"TRAINING RESULTS ({TARGET_METHOD})")
    print("="*80)
    print(json.dumps(results, indent=2))

In [ ]:
# Display results as table if available
if results_df is not None and not results_df.empty:
    print("\n" + "-"*80)
    print("METRICS SUMMARY")
    print("-"*80)
    print(results_df.to_string())
elif results:
    print("\n" + "-"*80)
    print("METRICS SUMMARY")
    print("-"*80)
    print(json.dumps(results, indent=2))

## 7. Save Results to Google Drive (if on Colab)

In [ ]:
if IN_COLAB:
    import shutil
    from pathlib import Path
    
    print(f"[Sync] Copying {TARGET_METHOD} results to Google Drive...")
    
    drive_method_dir = Path(f'/content/drive/MyDrive/taxi_demand_results/{TARGET_METHOD}')
    drive_method_dir.mkdir(parents=True, exist_ok=True)
    
    local_method_ckpt = Path(f'checkpoints/{TARGET_METHOD}')
    if local_method_ckpt.exists():
        dst_ckpt = drive_method_dir / 'checkpoints'
        if dst_ckpt.exists():
            shutil.rmtree(dst_ckpt)
        shutil.copytree(local_method_ckpt, dst_ckpt)
        print(f"  [OK] checkpoints/{TARGET_METHOD} synced")
    
    local_summary = Path('reports/training_summary.csv')
    if local_summary.exists():
        shutil.copy(local_summary, drive_method_dir / f'training_summary_{TARGET_METHOD}.csv')
        print(f"  [OK] training_summary_{TARGET_METHOD}.csv synced")
    
    local_logs = Path('logs')
    if local_logs.exists():
        log_dir = drive_method_dir / 'logs'
        log_dir.mkdir(parents=True, exist_ok=True)
        for log_file in local_logs.glob('*.json'):
            try:
                shutil.copy(log_file, log_dir / log_file.name)
            except Exception:
                pass
    
    print(f"\n[OK] {TARGET_METHOD} results synced to Google Drive")
else:
    print("[Info] Running locally - method outputs saved to:")
    print(f"  - checkpoints/{TARGET_METHOD}/")
    print("  - reports/training_summary.csv")
    print("  - logs/")

## Summary

In [ ]:
print("\n" + "="*80)
print(f"TRAINING WORKFLOW COMPLETE ({TARGET_METHOD})")
print("="*80)

print("\n[Results] Output files:")
print(f"  - Checkpoints: checkpoints/{TARGET_METHOD}/")
print("  - Summary:     reports/training_summary.csv")
print("  - Metrics:     checkpoints/<method>/metrics.json and logs/*.json")

if IN_COLAB:
    print("\n[Files] Google Drive path:")
    print(f"  - /content/drive/MyDrive/taxi_demand_results/{TARGET_METHOD}/")
else:
    print("\n[Files] All results are in the project directory")

print("\nTraining complete!")